# TP 2 : détecter et compter la mouche de l'olive dans les pièges

**Objectif.** La mouche de l'olive (*Bactrocera oleae*) est le principal ravageur des
oliveraies méditerranéennes. On la surveille avec des pièges McPhail. Le nombre de
mouches capturées par semaine décide du traitement. Vous allez entraîner un
**détecteur** (YOLO) qui encadre chaque insecte, distingue femelles, mâles et pyrales,
puis **compte** les mouches.

**Différence avec le TP 1.** On ne classe plus une image entière. On localise
**plusieurs objets petits** dans chaque image, avec une boîte et une classe chacun.

**Ce que vous allez apprendre.**

1. Lire des annotations COCO, regarder la taille des objets.
2. Convertir au format YOLO (exercice de géométrie des boîtes).
3. Entraîner YOLO11, lire précision, rappel, mAP.
4. Passer de la détection au **comptage**, la mesure qui compte pour l'agriculteur.
5. Tester sur d'autres pièges (Grèce) : généralisation.

**Données** (une fois, depuis la racine du projet) :

```bash
uv run agridrone data download olive_fly_smartmcphail olive_fly_dirt_coco olive_fly_digital_mcphail
```

Licence : Ultralytics (YOLO) est sous AGPL-3.0. C'est sans problème pour ce projet
académique.

## 0. Mise en route

In [ ]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch

# Racine du projet : le dossier qui contient pyproject.toml (le notebook peut être lancé
# depuis tp/ ou tp/corriges/).
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data" / "interim"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
# Sous Windows et macOS, les DataLoader à plusieurs processus ne fonctionnent pas avec des
# classes définies dans un notebook : on charge les images dans le processus principal.
NUM_WORKERS = 2 if sys.platform.startswith("linux") else 0
print(f"Projet : {ROOT}\nMatériel : {DEVICE}")

import json
import shutil
from collections import Counter, defaultdict

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

OUT = ROOT / "runs" / "tp2"
OUT.mkdir(parents=True, exist_ok=True)
YOLO_DIR = ROOT / "data" / "processed" / "tp2_mouche"

MODE_RAPIDE = DEVICE == "cpu"
EPOCHS = 10 if MODE_RAPIDE else 80
IMGSZ = 640 if MODE_RAPIDE else 1024    # taille des images vues par le réseau
BATCH = 8 if MODE_RAPIDE else 16
DEVICE_YOLO = 0 if DEVICE == "cuda" else DEVICE
print(f"MODE_RAPIDE={MODE_RAPIDE}, EPOCHS={EPOCHS}, IMGSZ={IMGSZ}")

## 1. Les données : SmartMcPhail

210 photos prises dans des pièges McPhail connectés, en oliveraies commerciales.
Chaque insecte est encadré et étiqueté `Female`, `Male` (mouche de l'olive) ou
`Glyphode` (pyrale du jasmin, autre ravageur). L'autrice fournit un découpage
train / valid / test, et elle a vérifié qu'aucune photo ne se trouve dans deux
découpages.

Format COCO : un fichier JSON avec la liste des `images`, des `annotations` (boîte
`bbox = [x, y, largeur, hauteur]` en pixels, coin haut-gauche) et des `categories`.

In [ ]:
SMART = DATA / "olive_fly_smartmcphail" / "FinalDataset.coco" / "FinalDataset.coco"

def lire_coco(fichier):
    """Une ligne par boîte : image, taille de l'image, classe, boîte COCO."""
    d = json.loads(Path(fichier).read_text(encoding="utf-8"))
    categories = {c["id"]: c["name"] for c in d["categories"]}
    images = {im["id"]: im for im in d["images"]}
    lignes = [{
        "image": images[a["image_id"]]["file_name"],
        "W": images[a["image_id"]]["width"], "H": images[a["image_id"]]["height"],
        "classe": categories[a["category_id"]],
        "x": a["bbox"][0], "y": a["bbox"][1], "w": a["bbox"][2], "h": a["bbox"][3],
    } for a in d["annotations"]]
    return pd.DataFrame(lignes), [im["file_name"] for im in d["images"]]

boites = {}
for split in ("train", "valid", "test"):
    boites[split], _ = lire_coco(SMART / split / "_annotations.coco.json")
    boites[split]["split"] = split
toutes = pd.concat(boites.values(), ignore_index=True)
pd.crosstab(toutes["classe"], toutes["split"], margins=True)

### Exercice 1.1 : quelle taille font les insectes ?

Ajoutez une colonne `taille_rel` = largeur de la boîte / largeur de l'image, tracez son
histogramme, puis calculez la taille médiane d'un insecte en pixels **après**
redimensionnement de l'image à `IMGSZ` pixels de côté (le plus grand côté).

❓ Un détecteur a du mal en dessous d'environ 10 pixels. Quelle valeur d'`IMGSZ`
conseillez-vous ?

In [ ]:
# Corrigé
toutes["taille_rel"] = toutes["w"] / toutes["W"]
toutes["taille_rel"].plot.hist(bins=40, figsize=(5, 3))
plt.xlabel("largeur de la boîte / largeur de l'image"); plt.show()
for taille in (640, 1024, 1280):
    px = (toutes["w"] / toutes[["W", "H"]].max(axis=1) * taille).median()
    print(f"IMGSZ={taille} : un insecte fait en médiane {px:.0f} px de large")

### Exercice 1.2 : afficher une image avec ses boîtes

Écrivez `afficher(split, image)`, qui dessine l'image et ses boîtes : un rectangle
`patches.Rectangle((x, y), w, h)` par boîte, avec une couleur par classe.

In [ ]:
# Corrigé
COULEURS = {"Female": "#e4572e", "Male": "#17bebb", "Glyphode": "#ffc914"}

def afficher(split, image):
    fig, ax = plt.subplots(figsize=(6, 8))
    ax.imshow(Image.open(SMART / split / image))
    for _, b in boites[split][boites[split]["image"] == image].iterrows():
        ax.add_patch(patches.Rectangle((b["x"], b["y"]), b["w"], b["h"], fill=False,
                                       lw=1.5, color=COULEURS.get(b["classe"], "white")))
    ax.set_title(image[:40]); ax.axis("off")
    ax.legend(handles=[patches.Patch(color=c, label=k) for k, c in COULEURS.items()], loc="lower right")
    plt.show()

afficher("train", boites["train"]["image"].iloc[0])

❓ Comment distinguez-vous une femelle d'un mâle ? (Indice : l'ovipositeur au bout de
l'abdomen.) Ce détail est-il visible à la résolution de travail du réseau ?

## 2. Convertir au format YOLO

YOLO attend, pour chaque image, un fichier `.txt` avec une ligne par objet :

```
classe x_centre y_centre largeur hauteur
```

Les quatre nombres sont **normalisés** entre 0 et 1 par la taille de l'image. COCO
donne `x, y` = coin haut-gauche, en pixels.

### Exercice 2.1

Complétez `coco_vers_yolo` : écrivez les étiquettes et copiez les images dans
`YOLO_DIR/images/<split>` et `YOLO_DIR/labels/<split>`. La catégorie
`Gliphode-Male-Female` est une super-catégorie vide de Roboflow : ignorez-la.

In [ ]:
# Corrigé
CLASSES = ["femelle", "male", "pyrale"]
COCO_VERS_ID = {"Female": 0, "Male": 1, "Glyphode": 2}

def coco_vers_yolo(json_coco, dossier_images, split):
    d = json.loads(Path(json_coco).read_text(encoding="utf-8"))
    categories = {c["id"]: c["name"] for c in d["categories"]}
    par_image = defaultdict(list)
    for a in d["annotations"]:
        par_image[a["image_id"]].append(a)
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
    for im in d["images"]:
        W, H = im["width"], im["height"]
        lignes = []
        for a in par_image[im["id"]]:
            nom = categories[a["category_id"]]
            if nom not in COCO_VERS_ID:
                continue
            x, y, w, h = a["bbox"]
            lignes.append(f"{COCO_VERS_ID[nom]} {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")
        nom_image = Path(im["file_name"])
        shutil.copy2(dossier_images / nom_image, YOLO_DIR / "images" / split / nom_image.name)
        (YOLO_DIR / "labels" / split / f"{nom_image.stem}.txt").write_text("\n".join(lignes), encoding="utf-8")

for split_coco, split_yolo in [("train", "train"), ("valid", "val"), ("test", "test")]:
    coco_vers_yolo(SMART / split_coco / "_annotations.coco.json", SMART / split_coco, split_yolo)

In [ ]:
# Fourni : fichier de description du jeu pour YOLO, et vérification.
DATA_YAML = YOLO_DIR / "data.yaml"
DATA_YAML.write_text(
    f"path: {YOLO_DIR.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
    + "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES)), encoding="utf-8")
for split in ("train", "val", "test"):
    n_img = len(list((YOLO_DIR / "images" / split).iterdir()))
    n_box = sum(len(f.read_text().split("\n")) for f in (YOLO_DIR / "labels" / split).glob("*.txt") if f.read_text())
    print(f"{split:5s} : {n_img} images, {n_box} boîtes")
print(DATA_YAML.read_text())

❓ Vérifiez une conversion à la main. Reprenez la première boîte d'une image, en
pixels, et retrouvez-la à partir de la ligne YOLO.

## 3. Entraîner YOLO11

`yolo11n.pt` est la plus petite version (2,6 M de paramètres), pré-entraînée sur COCO
(80 classes d'objets du quotidien). Ultralytics s'occupe de la boucle, des
augmentations (mosaïque, couleurs, échelles) et des courbes.

Durée indicative : sur CPU en `MODE_RAPIDE`, quelques minutes par époque ; sur GPU,
80 époques en 1024 px prennent de l'ordre d'une demi-heure (non mesuré ici).

In [ ]:
from ultralytics import YOLO

modele = YOLO("yolo11n.pt")   # téléchargé automatiquement la première fois
resultats = modele.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE_YOLO,
    seed=SEED, workers=NUM_WORKERS, project=str(OUT), name="mouche", exist_ok=True,
    plots=True, verbose=False,
)
MEILLEUR = OUT / "mouche" / "weights" / "best.pt"
print(MEILLEUR)

## 4. Évaluer la détection

- **Précision** : part des boîtes prédites qui sont de vrais insectes de la bonne
  classe.
- **Rappel** : part des vrais insectes trouvés.
- **mAP50** : précision moyenne, une boîte comptant comme juste si elle recouvre la
  vraie à au moins 50 % (IoU ≥ 0,5). **mAP50-95** fait de même avec des recouvrements
  exigés de 50 à 95 %.

### Exercice 4.1

Évaluez `MEILLEUR` sur le découpage `test` (`YOLO(MEILLEUR).val(..., split="test")`),
puis affichez précision, rappel, mAP50 et mAP50-95 par classe.

In [ ]:
# Corrigé
metriques = YOLO(MEILLEUR).val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
                               device=DEVICE_YOLO, plots=True, project=str(OUT), name="test",
                               exist_ok=True, verbose=False)
tableau = pd.DataFrame({
    "précision": metriques.box.p, "rappel": metriques.box.r,
    "mAP50": metriques.box.ap50, "mAP50-95": metriques.box.ap,
}, index=[metriques.names[int(i)] for i in metriques.box.ap_class_index]).round(3)
tableau.loc["toutes"] = [metriques.box.mp, metriques.box.mr, metriques.box.map50, metriques.box.map]
tableau.round(3)

❓ Ouvrez `runs/tp2/test/confusion_matrix.png`. Le détecteur confond-il surtout
femelles et mâles, ou rate-t-il des insectes (colonne « background ») ?

## 5. Du détecteur au comptage

L'agriculteur n'a pas besoin de boîtes. Il a besoin du **nombre de mouches** par
piège. On compte les détections de femelles et de mâles au-dessus d'un seuil de
confiance, puis on compare au vrai nombre.

### Exercice 5.1

Pour chaque image du test, comptez les mouches prédites (classes 0 et 1, confiance ≥
`CONF`) et les vraies (fichiers d'étiquettes). Calculez l'erreur absolue moyenne et
tracez prédit contre vrai.

In [ ]:
# Corrigé
CONF = 0.25
detecteur = YOLO(MEILLEUR)
comptes = []
for image in sorted((YOLO_DIR / "images" / "test").iterdir()):
    r = detecteur.predict(str(image), conf=CONF, imgsz=IMGSZ, device=DEVICE_YOLO, verbose=False)[0]
    predit = int(sum(int(c) in (0, 1) for c in r.boxes.cls.tolist()))
    etiquettes = (YOLO_DIR / "labels" / "test" / f"{image.stem}.txt").read_text().split("\n")
    vrai = sum(1 for l in etiquettes if l and int(l.split()[0]) in (0, 1))
    comptes.append({"image": image.name, "vrai": vrai, "predit": predit})
comptes = pd.DataFrame(comptes)
mae = (comptes["predit"] - comptes["vrai"]).abs().mean()
print(f"erreur absolue moyenne : {mae:.2f} mouche(s) par image ; "
      f"total vrai {comptes['vrai'].sum()}, total prédit {comptes['predit'].sum()}")
plt.scatter(comptes["vrai"], comptes["predit"], alpha=0.6)
m = comptes[["vrai", "predit"]].max().max()
plt.plot([0, m], [0, m], "k--", lw=1); plt.xlabel("mouches (vrai)"); plt.ylabel("mouches (prédit)")
plt.title("Comptage sur le test"); plt.show()

❓ Faites varier `CONF` (0,1 ; 0,25 ; 0,5). Comment évoluent l'erreur de comptage et le
total ? Quel seuil choisiriez-vous pour décider d'un traitement ?

## 6. Le vrai test : d'autres pièges

### 6.1 DIRT (Grèce, 2018)

291 images de pièges McPhail, avec des insectes flottant dans le liquide. Tous sont
annotés en polygones, sous une seule classe `insect`. On évalue donc la capacité à
**trouver les insectes**, sans distinguer les classes (`single_cls=True`).

### Exercice 6.1

Convertissez DIRT au format YOLO, avec une seule classe et un découpage `test`
seulement. Les annotations COCO contiennent aussi `bbox` : utilisez-la. Évaluez
ensuite le détecteur avec `single_cls=True`.

In [ ]:
# Corrigé
DIRT = DATA / "olive_fly_dirt_coco" / "DIRT" / "DIRT"
DIRT_YOLO = ROOT / "data" / "processed" / "tp2_dirt"
d = json.loads((DIRT / "instances_default.json").read_text(encoding="utf-8"))
par_image = defaultdict(list)
for a in d["annotations"]:
    par_image[a["image_id"]].append(a)
(DIRT_YOLO / "images" / "test").mkdir(parents=True, exist_ok=True)
(DIRT_YOLO / "labels" / "test").mkdir(parents=True, exist_ok=True)
for im in d["images"]:
    W, H = im["width"], im["height"]
    lignes = [f"0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}"
              for x, y, w, h in (a["bbox"] for a in par_image[im["id"]])]
    shutil.copy2(DIRT / im["file_name"], DIRT_YOLO / "images" / "test" / Path(im["file_name"]).name)
    (DIRT_YOLO / "labels" / "test" / f"{Path(im['file_name']).stem}.txt").write_text("\n".join(lignes), encoding="utf-8")
yaml_dirt = DIRT_YOLO / "data.yaml"
yaml_dirt.write_text(f"path: {DIRT_YOLO.as_posix()}\ntrain: images/test\nval: images/test\ntest: images/test\n"
                     "names:\n  0: insecte\n", encoding="utf-8")

metriques_dirt = YOLO(MEILLEUR).val(data=str(yaml_dirt), split="test", single_cls=True, imgsz=IMGSZ,
                                    batch=BATCH, device=DEVICE_YOLO, project=str(OUT), name="dirt",
                                    exist_ok=True, verbose=False)
print(f"DIRT : précision {metriques_dirt.box.mp:.3f}, rappel {metriques_dirt.box.mr:.3f}, "
      f"mAP50 {metriques_dirt.box.map50:.3f}")

❓ Le score s'effondre-t-il ? Regardez quelques images DIRT. Qu'est-ce qui change :
liquide, reflets, densité d'insectes, cadrage ?

### 6.2 Piège McPhail numérique (Leucade, 2022)

125 images, sans boîtes, tirées au hasard dans six pièges et cinq niveaux de capture.
Le fichier `counts.csv` donne pour chaque image (`image_file`) le nombre total de
mouches visibles, recompté à l'aveugle par un expert (`expert_count`). C'est
exactement la mesure utile au terrain. La colonne `pipeline_count` est un incrément
journalier : ne pas la comparer à vos comptages.

### Exercice 6.2

Prédisez le nombre de mouches de chaque image et comparez-le au comptage de l'expert :
erreur absolue moyenne, nuage de points, puis erreur par niveau de capture
(`density_bin`).

In [ ]:
# Corrigé
DMT = DATA / "olive_fly_digital_mcphail" / "Zenodo_DMT_validation_dataset"
experts = pd.read_csv(DMT / "counts.csv")
predits = []
for nom in experts["image_file"]:
    res = detecteur.predict(str(DMT / "images" / nom), conf=CONF, imgsz=IMGSZ,
                            device=DEVICE_YOLO, verbose=False)[0]
    predits.append(int(sum(int(c) in (0, 1) for c in res.boxes.cls.tolist())))
dmt = experts.assign(predit=predits).rename(columns={"expert_count": "expert"})
dmt["erreur"] = (dmt["predit"] - dmt["expert"]).abs()
print(f"{len(dmt)} images ; erreur absolue moyenne {dmt['erreur'].mean():.2f}")
display(dmt.groupby("density_bin")[["expert", "predit", "erreur"]].mean().round(1))
plt.scatter(dmt["expert"], dmt["predit"], alpha=0.6)
m = dmt[["expert", "predit"]].max().max()
plt.plot([0, m], [0, m], "k--", lw=1); plt.xlabel("comptage de l'expert"); plt.ylabel("mouches prédites")
plt.title("Leucade : comptage hors source"); plt.show()

## 7. Pour aller plus loin

1. **Petits objets** : découpez chaque image en tuiles de 640 px qui se recouvrent,
   détectez sur chaque tuile, puis fusionnez les boîtes. La bibliothèque SAHI fait
   cela : `uv pip install sahi`. Le rappel augmente-t-il ?
2. **Plus de données** : ajoutez le jeu croate de Zadar (`olive_fly_zadar`, boîtes
   Pascal VOC) à l'entraînement. Écrivez la conversion VOC → YOLO.
3. **Un modèle plus grand** : `yolo11s.pt`. Que gagne-t-on en mAP, et que perd-on en
   vitesse ?
4. **Vers le drone** : les pièges jaunes englués peuvent être photographiés par un
   drone qui survole l'oliveraie. Quelle résolution au sol faudrait-il pour voir une
   mouche de 5 mm sur au moins 10 pixels ? (`GSD = taille / pixels`)

## Compte rendu attendu

Une à deux pages :
- effectifs et taille des insectes ;
- choix d'`IMGSZ` justifié ;
- tableau des métriques du test ;
- erreur de comptage et seuil choisi ;
- résultats hors source (DIRT, Leucade) ;
- vos réponses aux ❓.